# DG-4 walkthrough — failure-envelope via the parity-aware $r_4$ ratio

> **⚠️ VERDICT SUPERSEDED — read before relying on the outputs below.**
>
> The v0.1.1 PASS verdict shown in this notebook (tag `v0.5.0`, 2026-05-06) was **superseded on review** the same day for two HIGH-severity defects:
>
> 1. **Path B order-4 extraction is in the wrong picture.** [`benchmarks/numerical_tcl_extraction.py`](../benchmarks/numerical_tcl_extraction.py) computes `L_2 = dΛ_2/dt` and `L_4 = dΛ_4/dt - L_2 Λ_2`, which is the order-4 expansion of `L_t = Λ̇_t Λ_t⁻¹` only when the closed-system map `Λ_0 = id` (interaction picture, or `H_S = 0`). The pipeline reconstructs raw Schrödinger-picture maps from `exact_finite_env`, only subtracts `Λ_0` as a polynomial-fit baseline, and does *not* apply the `Λ_0⁻¹` similarity or the `L_0 Λ_n Λ_0⁻¹` correction. For σ_x thermal (`H_S = (ω/2) σ_z`, `ω = 1`), neither omitted piece vanishes, so the extracted `‖L_n^dis‖_F` is not the picture-invariant quantity the metric is supposed to be.
> 2. **PASS predicate trivially satisfied for two of four perturbations.** The runner requires `r_4 > 1` to be stable under all four reproducibility perturbations including `upper_cutoff_factor ∈ {20, 40}`. But Path B's `_path_b_evaluate` records that `exact_finite_env` ignores the threaded `quadrature_kwargs`, so for those two perturbations `r_4_perturbed == r_4_baseline` trivially — the predicate adds zero independent evidence. The result JSON marks both checks as `operational_in_path_b: false` while still recording `verdict: PASS`. Honest caveat ≠ predicate satisfaction.
>
> Plus a MEDIUM audit-completeness gap (per-α + per-perturbation `r_4` not persisted in the result JSON) and a LOW version-semantics drift (CHANGELOG `[0.5.0]` vs package metadata `0.3.0.dev0`).
>
> The supersedure is recorded in [`logbook/2026-05-06_dg-4-pass-path-b-superseded.md`](../logbook/2026-05-06_dg-4-pass-path-b-superseded.md). The `v0.5.0` git tag is left as immutable history; the verdict it anchored is downgraded on the record. **What still stands**: the runner is reachable on D1 v0.1.1 and produces a structured CardResult; the metric design (parity-aware even-order ratio) and the σ_x thermal Path B fixture are valid as a route-of-attack; the notebook demonstrates the pipeline end-to-end. **What does not stand**: the PASS verdict, the `convergence_failure` classifications, and any citation of the repository for a DG-4 failure-envelope identification at this version. v0.1.2 supersedure work: repair Path B picture handling (regression test on a fixture with `H_S ≠ 0`), thread `upper_cutoff_factor` into `exact_finite_env`'s `omega_max_factor` so the perturbation is operational, persist per-α + per-perturbation `r_4` in the result JSON, freeze D1 v0.1.2 superseding v0.1.1, re-run.

---

**Decision Gate:** DG-4 (failure envelope: cause-labelled, reproducible failure regime).
**Verdict status:** v0.1.1 PASS at tag `v0.5.0` **superseded on review** (2026-05-06); v0.1.2 supersedure pending.
**Card exercised:** [D1 v0.1.1](../benchmarks/benchmark_cards/D1_failure-envelope-convergence_v0.1.1.yaml).
**Authoritative status:** [`docs/validity_envelope.md`](../docs/validity_envelope.md).

This notebook reproduces the v0.1.1 pipeline on a reduced fixture for pedagogical purposes — the cells run in seconds and show how the Path B route, the runner, and the metric integrate end-to-end. Read the supersedure banner above before treating any of the numbers as DG-4 authoritative.

The full frozen run uses `n_bath_modes=4`, `n_levels_per_mode=3`, a 200-point time grid and a 20-point `coupling_strength` sweep, and takes ~150 s on a laptop. The reduced fixture below uses `n_bath_modes=2`, `n_levels_per_mode=2`, an 11-point time grid, and a 4-point sweep so the cells run in seconds — the upper half of the sweep range still classifies as `convergence_failure_candidate` under the (currently superseded) v0.1.1 metric, which was the basis of the v0.5.0 PASS that was downgraded on review.

DG-4 PASS condition (Sail v0.5 §9): identify at least one reproducible, cause-labelled failure regime. The metric is the parity-aware even-order ratio

$$
r_4(\alpha^2) \;=\; \alpha^2\, \frac{\langle \lVert L_4^{\text{dis}}\rVert\rangle_t}{\langle \lVert L_2^{\text{dis}}\rVert\rangle_t},
\qquad L_n^{\text{dis}} := L_n + i\,[K_n,\,\cdot\,].
$$

The card sweeps `coupling_strength = α²` across the frozen range. A swept point classifies as `convergence_failure` iff $r_4 > 1$ at baseline **and** stays $> 1$ under the v0.1.4 reproducibility perturbations (`upper_cutoff_factor`, `omega_c`).

> **Caveat — Path B vs Path A.** This notebook uses the *numerical* Richardson route (Path B) for the order-4 generator $L_4$. Path B carries a documented finite-environment extraction floor; analytic Path A (Companion Sec. IV closed form for $L_4$) remains preferred and pending. Independent of Path A vs B, the v0.1.1 Path B implementation is in the wrong picture (see banner above) and is being repaired in v0.1.2.


## 0. Setup

In [1]:
import copy

import numpy as np
import yaml

import cbg
from reporting import benchmark_card as bc
from benchmarks import exact_finite_env, numerical_tcl_extraction as nte

print(f"cbg.__version__       = {cbg.__version__}")
print(f"cbg.__sail_version__  = {cbg.__sail_version__}")
print(f"cbg.__ledger_anchor__ = {cbg.__ledger_anchor__}")

cbg.__version__       = 0.3.0.dev0
cbg.__sail_version__  = 0.5
cbg.__ledger_anchor__ = CL-2026-005_v0.4


## 1. Load the frozen card

Card D1 v0.1.1 freezes the σ_x thermal model with the parity-aware ratio metric. We load it via the schema-aware loader; this also validates the card against `SCHEMA.md v0.1.3`.

In [2]:
from pathlib import Path

D1_PATH = Path(cbg.__file__).parent.parent / "benchmarks" / "benchmark_cards" / "D1_failure-envelope-convergence_v0.1.1.yaml"
card = bc.load_card(D1_PATH)
print(f"card_id           = {card.card_id} {card.version}")
print(f"dg_target         = {card.dg_target}")
print(f"model             = {card.model}")
print(f"target_observable = {card.frozen_parameters['comparison']['target_observable']}")
print(f"threshold         = {card.threshold}")
sweep_range = card.frozen_parameters["sweep"]["sweep_range"]
print(f"sweep             = coupling_strength: {sweep_range['start']} → {sweep_range['end']} ({sweep_range['n_points']} pts, {sweep_range['scheme']})")

card_id           = D1 v0.1.1
dg_target         = DG-4
model             = spin_boson_sigma_x
target_observable = r_4 = <||L_4^dissipator||>_t / <||L_2^dissipator||>_t
threshold         = 1.0
sweep             = coupling_strength: 0.05 → 1.0 (20 pts, log_uniform)


## 2. Path B coefficient computation — $\langle\lVert L_2^{\text{dis}}\rVert\rangle_t$ and $\langle\lVert L_4^{\text{dis}}\rVert\rangle_t$

Path B fits the alpha-power expansion
$$
\Lambda_t(\alpha) \;=\; \Lambda_0 \;+\; \alpha^2\,\Lambda_2 \;+\; \alpha^4\,\Lambda_4 \;+\; O(\alpha^6)
$$
to finite-environment process tomography, takes time derivatives to recover $L_2$ and $L_4$, and reports the time-averaged Liouville-Frobenius norms of the dissipator residuals.

These are *coefficients* in the alpha-expansion, so they are independent of the swept `coupling_strength = α²` value and a single fit covers the entire alpha-sweep.

In [3]:
model_spec = copy.deepcopy(card.frozen_parameters["model"])
# Reduced fixture for fast walkthrough: smaller bath, coarser time grid.
t_grid = np.linspace(0.0, 1.0, 11)
alpha_values = (0.01, 0.02, 0.03)

coeffs = nte.path_b_dissipator_norm_coefficients(
    builder=exact_finite_env.build_spin_boson_sigma_x_thermal_total,
    model_spec=model_spec,
    t_grid=t_grid,
    alpha_values=alpha_values,
    builder_kwargs={"n_bath_modes": 2, "n_levels_per_mode": 2},
)

print(f"fit_relative_residual    = {coeffs.fit_relative_residual:.3e}")
print(f"<||L_2^dis||>_t          = {coeffs.l2_avg:.3e}")
print(f"<||L_4^dis||>_t          = {coeffs.l4_avg:.3e}")
print(f"coefficient ratio l4/l2  = {coeffs.l4_avg / coeffs.l2_avg:.3e}")

fit_relative_residual    = 2.026e-07
<||L_2^dis||>_t          = 9.720e+00
<||L_4^dis||>_t          = 6.145e+01
coefficient ratio l4/l2  = 6.322e+00


## 3. Per-α evaluation of the parity-aware $r_4$ ratio

The runner scales the $\ell_4 / \ell_2$ coefficient ratio by $\alpha^2$ at each swept value of `coupling_strength`. The threshold is $1.0$: any $\alpha$ with $r_4 > 1$ is a *failing candidate*.

In [4]:
sweep_alphas_sq = np.geomspace(sweep_range["start"], sweep_range["end"], num=4)
ratio_coeff = coeffs.l4_avg / coeffs.l2_avg
r_4_baseline = sweep_alphas_sq * ratio_coeff

print("alpha²       r_4_baseline     classification")
print("-" * 50)
for a2, r4 in zip(sweep_alphas_sq, r_4_baseline):
    label = "convergence_failure_candidate" if r4 > 1.0 else "passing"
    print(f"{a2:8.4f}    {r4:10.3e}     {label}")

alpha²       r_4_baseline     classification
--------------------------------------------------
  0.0500     3.161e-01     passing
  0.1357     8.580e-01     passing
  0.3684     2.329e+00     convergence_failure_candidate
  1.0000     6.322e+00     convergence_failure_candidate


## 4. Running the full DG-4 verdict via `run_card`

The runner pipeline (`reporting.benchmark_card._run_dg4_sweep`) does what we did manually above, and additionally re-runs Path B under each reproducibility perturbation (`upper_cutoff_factor` ∈ {20, 40} via the runner's quadrature path; `omega_c` ∈ {9, 11} via direct model-spec mutation). A failing candidate is reclassified `convergence_failure` iff $r_4 > 1$ stays under all four perturbations; otherwise `truncation_artefact`.

We use a reduced fixture with `n_points_sweep=4`, `t_end=1.0`, `n_t=11`, `n_bath_modes=2`, `n_levels_per_mode=2`. The full frozen card uses `n_points_sweep=20`, `t_end=20.0`, `n_t=200`, `n_bath_modes=4`, `n_levels_per_mode=3`; the verdict on the full card is recorded in [`benchmarks/results/D1_failure-envelope-convergence_v0.1.1_result.json`](../benchmarks/results/D1_failure-envelope-convergence_v0.1.1_result.json) and in the [DG-4 PASS logbook entry](../logbook/2026-05-06_dg-4-pass-path-b.md).

In [5]:
data = copy.deepcopy(yaml.safe_load(open(D1_PATH)))
data["frozen_parameters"]["numerical"]["time_grid"]["t_end"] = 1.0
data["frozen_parameters"]["numerical"]["time_grid"]["n_points"] = 11
data["frozen_parameters"]["sweep"]["sweep_range"]["n_points"] = 4
data["frozen_parameters"]["numerical"]["path_b"] = {
    "alpha_values": [0.01, 0.02, 0.03],
    "n_bath_modes": 2,
    "n_levels_per_mode": 2,
}
bc.validate_card_data(data)
reduced_card = bc._data_to_card(data)

result = bc.run_card(reduced_card)
print(f"verdict          = {result.verdict}")
print(f"runner_version   = {result.runner_version}")
print(f"# test_cases     = {len(result.test_case_results)}")
tcr = result.test_case_results[0]
print(f"test_case        = {tcr.name}")
print(f"max baseline r_4 = {tcr.error:.3e}  (threshold = {tcr.threshold})")

verdict          = PASS
runner_version   = 0.1.0
# test_cases     = 1
test_case        = dg4_failure_envelope_sweep
max baseline r_4 = 6.322e+00  (threshold = 1.0)


The runner notes record the per-α classifications, the Path B fit residual, the time-averaged norm coefficients, and the Path B finite-env caveat. The relevant lines are reproduced below.

In [6]:
relevant_lines = [
    line for line in result.notes.splitlines()
    if any(token in line for token in (
        "verdict", "Verdict", "PASS", "FAIL", "CONDITIONAL",
        "convergence_failure", "truncation_artefact",
        "Path B", "finite-env", "alpha_crit",
    ))
]
print("\n".join(relevant_lines[:20]))

L_4 source: Path B numerical Richardson extraction.
Path B params: alpha_values=(0.01, 0.02, 0.03), n_bath_modes=2, n_levels_per_mode=2.
alpha_crit (interpolated, log-linear): 1.4946e-01.
Per-alpha classifications: passing=2, convergence_failure=2.
Path B carries a finite-env extraction floor (sigma_z thermal zero-oracle ~few * 1e-2 at the default truncation per the 2026-05-06 pilot); verdicts are subject to this documented uncertainty until the analytic Path A (Companion Sec. IV) lands.


## 5. Outcome (verdict superseded — see banner at top)

On the reduced fixture, the upper two of four swept `coupling_strength` points classify as `convergence_failure_candidate` under the v0.1.1 metric, and the runner interpolates `α_crit` inside the swept range (between the second and third swept points). Under the v0.1.1 predicate that was sufficient for `verdict = PASS`. The full frozen run pushed `α_crit` below the lowest swept point with maximum baseline `r_4 ≈ 46.51`, and was the basis of the 2026-05-06 PASS verdict at tag `v0.5.0`.

**That verdict has been superseded on review** for the two HIGH-severity defects called out in the banner at the top of this notebook (Path B order-4 extraction in the wrong picture for `H_S ≠ 0`; PASS predicate trivially satisfied for two of four perturbations) plus a MEDIUM audit-completeness gap. The numbers above are produced by the v0.1.1 pipeline as-committed; without the picture repair we cannot say whether a corrected extraction would still classify these points as failing — the magnitudes will move.

The cause label `convergence failure` from the §4 cause-label taxonomy in [`docs/benchmark_protocol.md`](../docs/benchmark_protocol.md) was the target of the v0.1.1 verdict; it remains the target of the pending v0.1.2 supersedure. The repair work is recorded in the supersedure logbook entry [`2026-05-06_dg-4-pass-path-b-superseded.md`](../logbook/2026-05-06_dg-4-pass-path-b-superseded.md).

> **Citation note.** Since the v0.1.1 PASS verdict is superseded, the repository may **not** currently be cited for the existence of a DG-4 failure-envelope identification or for any convergence-breakdown claim. CL-2026-005 v0.4 Entry 2 remains COMPATIBLE *scope-limited* per [`docs/do_not_cite_as.md`](../docs/do_not_cite_as.md), and Entry 2's convergence-reliability question remains unresolved at this version.